# imports

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.special import gammaln
from openmm import *
from openmm.app import *
from openmm.unit import *
import mdtraj as md
import h5py

## General functions

In [ ]:
# ============================================================
# TOPOLOGY GRAPH UTILITIES
# ============================================================

def get_downstream_atoms(topology, atom2, atom3):
    """
    Find all atoms downstream of the bond atom2-atom3.

    The bond atom2 -> atom3 defines the rotation axis.
    Everything connected to atom3 without crossing back
    through atom2 will be rotated.
    """

    neighbors = {atom.index: [] for atom in topology.atoms()}

    for bond in topology.bonds():
        i = bond.atom1.index
        j = bond.atom2.index

        neighbors[i].append(j)
        neighbors[j].append(i)

    visited = {atom2}
    queue = [atom3]

    downstream = []

    while queue:
        current = queue.pop(0)

        if current not in downstream:
            downstream.append(current)

        for neighbor in neighbors[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return downstream


# ============================================================
# ROTATION MATH
# ============================================================

def rotate_points(points, origin, axis, theta):
    """
    Rotate coordinates around an axis using Rodrigues' formula.
    """

    axis = axis / np.linalg.norm(axis)

    shifted = points - origin

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    rotated = (
        shifted * cos_t
        + np.cross(axis, shifted) * sin_t
        + axis * np.sum(shifted * axis, axis=1)[:, None] * (1 - cos_t)
    )

    return rotated + origin


# ============================================================
# DIHEDRAL ROTATION ON VACUUM STRUCTURE
# ============================================================

def init_rotate_dihedral(
    topology,
    positions,
    dihedral_atoms,
    target_angle_deg
):
    """
    Rotate a dihedral BEFORE solvation/system creation.

    Parameters
    ----------
    topology : OpenMM topology

    positions : OpenMM Quantity array

    dihedral_atoms : list[int]
        [a1, a2, a3, a4]

    target_angle_deg : float
    """

    pos = positions.value_in_unit(nanometer).copy()
    pos = np.array(pos)
    a1, a2, a3, a4 = dihedral_atoms

    # --------------------------------------------------------
    # Compute current dihedral
    # --------------------------------------------------------

    traj = md.Trajectory(
        pos.reshape(1, -1, 3),
        md.Topology.from_openmm(topology)
    )

    current_angle = md.compute_dihedrals(
        traj,
        [dihedral_atoms]
    )[0][0]

    target_angle = np.radians(target_angle_deg)

    delta = target_angle - current_angle

    # --------------------------------------------------------
    # Rotation axis = bond a2-a3
    # --------------------------------------------------------

    axis = pos[a3] - pos[a2]
    axis /= np.linalg.norm(axis)

    # --------------------------------------------------------
    # Determine which atoms move
    # --------------------------------------------------------

    moving_atoms = get_downstream_atoms(
        topology,
        a2,
        a3
    )

    # Keep axis atom fixed
    moving_atoms = [i for i in moving_atoms if i != a3]

    # --------------------------------------------------------
    # Rotate coordinates
    # --------------------------------------------------------

    coords_to_rotate = pos[moving_atoms]

    rotated_coords = rotate_points(
        coords_to_rotate,
        origin=pos[a3],
        axis=axis,
        theta=delta
    )

    pos[moving_atoms] = rotated_coords

    return pos * nanometer

def numpy_dihedral_batch(coords: np.ndarray, atom_indices: list) -> np.ndarray:
    i, j, k, l = atom_indices
    p0 = coords[:, i]   # (n_frames, 3)
    p1 = coords[:, j]
    p2 = coords[:, k]
    p3 = coords[:, l]

    b0 = p0 - p1
    b1 = p2 - p1
    b2 = p3 - p2

    b1 = b1 / np.linalg.norm(b1, axis=1, keepdims=True)

    # Project onto plane perpendicular to b1
    v = b0 - (b0 * b1).sum(axis=1, keepdims=True) * b1
    w = b2 - (b2 * b1).sum(axis=1, keepdims=True) * b1

    # Per-frame dot products (NOT matrix multiplication)
    x = (v * w).sum(axis=1)
    y = (np.cross(b1, v) * w).sum(axis=1)

    return np.arctan2(y, x)

def _find_crossing_from_dataset(
    coord_dataset,           # open h5py Dataset object
    interval: int,
    phi_atoms: list,
    psi_atoms: list,
) -> int:
    """Internal version that works on an already-opened h5py dataset.

    Refreshes the dataset (SWMR), reads the last ``interval`` frames, computes
    φ/ψ with the vectorized batch dihedral, and returns ``removed_frames``.
    """
    # Force h5py to see the latest frames written by the running simulation
    coord_dataset.id.refresh()
        
    # 2. Use negative slicing [-interval:] to pull only the last interval frames
    # Shape of last_interval_positions will be (interval, n_atoms, 3)
    last_interval_positions = coord_dataset[-interval:]
    
    # 3. calculate angles for the last interval frames
    phis = numpy_dihedral_batch(last_interval_positions, phi_atoms)    # (interval,)
    psis = numpy_dihedral_batch(last_interval_positions, psi_atoms)    # (interval,
    # 4. Check which frames in the last interval are in the stable region
    stable_mask = angles_in_stable_region(phis, psis)
    first_entry = np.where(stable_mask)[0][0] #index of the FIRST stable frame in the last interval
    #5. Count how many frames after the first entry frame are in the stable region, we want to make sure there is only one stable frame in the last interval, which is the one we are shooting to, to ensure we are capturing the transition path accurately
    return interval - first_entry - 1 #number of frames AFTER the stable frame in the last interval


def numpy_dihedral(p0, p1, p2, p3):
    b0 = p0 - p1
    b1 = p2 - p1
    b2 = p3 - p2

    b1 = b1 / np.linalg.norm(b1)

    v = b0 - np.dot(b0, b1) * b1
    w = b2 - np.dot(b2, b1) * b1

    x = np.dot(v, w)
    y = np.dot(np.cross(b1, v), w)

    return np.arctan2(y, x)

def extract_original_transition_path(psis, phis):
    """Locate and extract the first valid, continuous transition path between two stable basins.

    This function scans matched time-series arrays of φ (phi) and ψ (psi) dihedral angles
    to find a sequence of frames representing a complete, uninterrupted transition from 
    Basin A to Basin B, or vice versa. 

    A valid transition path is defined by the following criteria:
    1. It begins strictly with exactly one frame in the initial stable basin.
    2. All intermediate frames reside in the unstable (transition) region.
    3. It ends strictly with exactly one frame in the opposite stable basin.
    4. Every frame within the entire path slice (entry to exit, inclusive) must have 
       a φ (phi) angle strictly less than 0.

    Parameters
    ----------
    psis : array_like of float
        A 1-D sequence of ψ (psi) dihedral angles (in radians) corresponding to a 
        molecular trajectory.
    phis : array_like of float
        A 1-D sequence of φ (phi) dihedral angles (in radians) corresponding to the 
        same molecular trajectory. Must be exactly the same length as `psis`.

    Returns
    -------
    tuple of (int, int) or None
        A tuple `(entry_index, exit_index)` representing the boundary indices of the 
        first discovered transition path. 
        - `entry_index` is the index of the last stable frame before the transition.
        - `exit_index` is the index of the first stable frame after the transition, 
          **plus one**. This makes the returned tuple directly usable for standard 
          exclusive Python slicing (e.g., `trajectory[entry_index : exit_index]`).
        If no valid transition path is found in the provided arrays, the function 
        implicitly returns `None`.

    Raises
    ------
    ValueError
        If the lengths of the `psis` and `phis` input sequences are not equal.

    Notes
    -----
    * **Basin Definitions:** 
      - Basin A is bounded by `psi > -π/4`.
      - Basin B is bounded by `psi < -8π/9`.
    * **Dependency:** This function relies on an external `angles_in_stable_region(phis, psis)` 
      callable that must return a boolean mask (True for stable, False for unstable) 
      of the same shape as the inputs.
    * **Directionality:** The function correctly identifies both forward (A → B) and 
      backward (B → A) transitions, returning the first one it encounters chronologically.
    """
    if len(psis) != len(phis):
        raise ValueError("Length of psis and phis must be the same.")
    psis = np.array(psis)
    phis = np.array(phis)
    stable_mask = angles_in_stable_region(phis, psis)
    indices_in_unstable = np.where(~stable_mask)[0]
    entry_index = np.where(stable_mask[indices_in_unstable-1], indices_in_unstable-1, None)
    exit_index = np.where(stable_mask[indices_in_unstable+1], indices_in_unstable+1, None)
    entry_index = entry_index[entry_index != None]
    exit_index = exit_index[exit_index != None]
    for j in entry_index:
        first_in_A = psis[j] > - np.pi/4 
        possible_exit = np.where(j < exit_index, exit_index, None)
        possible_exit = possible_exit[possible_exit != None]
        if not (stable_mask[j+1: possible_exit[0]]).any(): # Check if all frames between entry and exit are unstable
            last_in_B =   - np.pi * 8/9 > psis[possible_exit[0]] 
            if (first_in_A and last_in_B): # Check if we are transitioning between the two stable regions
                if (phis[j:  possible_exit[0]+1] < 0).all(): # Check if all frames between entry and exit have phi < 0
                    return j, possible_exit[0] + 1 
            else:
                first_in_B = psis[j] < - np.pi * 8/9
                last_in_A = - np.pi/4 < psis[possible_exit[0]]
                if (first_in_B and last_in_A):
                    if (phis[j:  possible_exit[0]+1] < 0).all(): # Check if all frames between entry and exit have phi < 0
                        return j, possible_exit[0] +1 # +1 to include the possible_exit[0] point // END INDEX
    print("No transition found!")
    

def angles_in_stable_region(phi, psi):
    """
    Define a simple criterion for whether a given (phi, psi) pair in radians is in a "stable" region.

    """
    check_array = np.array([phi, psi])
    return np.logical_and(check_array[0] < 0, np.logical_or(check_array[1] < -(np.pi * 8/9), check_array[1] > - (np.pi / 4) ))

def plot_ramachandran(traj, num=0, phi_atoms=None, psi_atoms=None, highlight_stable = False, phi_psi_traj = None):
    """Generate a basic Ramachandrom plot for a given trajectory.

    Parameters
    ----------
    traj
        An MDTraj trajectory object.
    phi_atoms
        A list of atom names (in order) to identify the phi angle.
        The defaults in MDTraj do not work for termini in CHARMM
        topologies, which can be fixed with this argument.
    psi_atoms
        A list of atom names (in order) to identify the psi angle.
        The defaults in MDTraj do not work for termini in CHARMM
        topologies, which can be fixed with this argument.

    """
    if phi_psi_traj is None:
        if phi_atoms is None:
            phis = md.compute_phi(traj)[1].ravel()
        else:
            phis = md.compute_dihedrals(
                traj, md.geometry.dihedral._atom_sequence(traj.topology, phi_atoms)[1]
            )
        if psi_atoms is None:
            psis = md.compute_psi(traj)[1].ravel()
        else:
            psis = md.compute_dihedrals(
                traj, md.geometry.dihedral._atom_sequence(traj.topology, psi_atoms)[1]
            )
    else:
        phis = phi_psi_traj[:,0]
        psis = phi_psi_traj[:,1]
    plt.close(num)
    fig = plt.figure(num=num, figsize=(12, 9))
    gs = fig.add_gridspec(2, 3)
    len_phis = len(phis)
    steps = np.arange(len_phis)
    # Ramachandran plot
    ax1 = fig.add_subplot(gs[:2, :2])
    ax1.scatter(phis, psis, c=range(len_phis), cmap='cool')
    ax1.axvline(0)
    ax1.axhline(- np.pi * 8/9, color="r", linestyle="--")
    ax1.axhline(- np.pi/4, color="r", linestyle="--")
    ax1.set_xlim(-np.pi, np.pi)
    ax1.set_ylim(-np.pi, np.pi)
    ax1.set_xticks(np.linspace(-np.pi, np.pi, 9))
    ax1.set_yticks(np.linspace(-np.pi, np.pi, 9))
    ax1.set_xlabel("Phi [rad]")
    ax1.set_ylabel("Psi [rad]")
    # ax1.set_aspect("equal", adjustable="datalim")
    # Phi(t) plot
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.plot(steps, phis, "k+")
    ax2.axhline(0)
    ax2.set_ylim(-np.pi, np.pi)
    ax2.set_yticks(np.linspace(-np.pi, np.pi, 9))
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Phi [rad]")
    # Psi(t) plot
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.plot(steps, psis, "k+")
    ax3.axhline(0)
    ax3.set_ylim(-np.pi, np.pi)
    ax3.set_yticks(np.linspace(-np.pi, np.pi, 9))
    ax3.set_xlabel("Step")
    ax3.set_ylabel("Psi [rad]")
    if highlight_stable:
        stable_mask = angles_in_stable_region(phis, psis)
        ax2.scatter(steps[stable_mask], phis[stable_mask], c='r', marker='+', s=50, zorder=3)
        ax3.scatter(steps[stable_mask], psis[stable_mask], c='r', marker='+', s=50, zorder=3)
    plt.tight_layout()
    plt.show()
    


# Shooting algorithm

In [ ]:
def copy_slice_h5_to_reporter(active_reporter, source_file, start_index, end_index):
    """Copy and optionally stitch trajectory slices from an HDF5 file into an active reporter.

    This function reads specific trajectory datasets from a source HDF5 file and 
    injects them directly into the underlying file writer of an active MDTraj-based 
    simulation reporter. 

    It supports two modes of operation based on the type of the index arguments:
    1. **Standard Mode (Integers):** Extracts a continuous block of frames from 
       `start_index` to `end_index`.
    2. **Stitching Mode (Lists):** Designed specifically for two-way Transition Path 
       Sampling (TPS). It takes two pairs of indices, extracts the first segment, 
       **reverses it in time**, and concatenates it with the second segment before 
       writing.

    Parameters
    ----------
    active_reporter : object
        An active OpenMM simulation reporter (e.g., an MDTraj `HDF5Reporter`). 
        It must possess an internal `_traj_file` attribute with a `write()` method 
        capable of accepting standard trajectory keyword arguments, as well as a 
        `flush()` method.
    source_file : str
        File path to the source HDF5 trajectory file from which frames are read.
    start_index : int or list of int
        The starting frame index for the slice. If a list is provided, it must 
        contain exactly two integers: `[backward_start, forward_start]`.
    end_index : int or list of int
        The ending frame index for the slice (exclusive). If a list is provided, 
        it must contain exactly two integers: `[backward_end, forward_end]`.

    Returns
    -------
    str
        A success message indicating completion. If integers were passed, it reports 
        the exact number of frames injected. If lists were passed, it returns `"Done!"`.

    Raises
    ------
    KeyError
        If the `source_file` is missing any of the strictly required datasets: 
        `'cell_angles'`, `'cell_lengths'`, `'coordinates'`, `'kineticEnergy'`, 
        `'potentialEnergy'`, `'temperature'`, `'time'`, or `'velocities'`.
    IndexError
        If `start_index` and `end_index` are lists but do not contain exactly two 
        elements corresponding to the two trajectory halves.

    Notes
    -----
    * In stitching mode, the first half is explicitly reversed (`[::-1]`). This 
      accommodates the backward integration phase of a TPS shooting move, ensuring 
      the final concatenated trajectory flows chronologically from Basin A to Basin B.
    * The function performs a hardcoded extraction of specific simulation properties. 
      Any custom datasets present in the source HDF5 file will be read into memory 
      but ignored during the `traj_writer.write()` step.
    * The injected frames are immediately flushed to disk via `traj_writer.flush()`.
    """
    sliced_data = {}
    with h5py.File(source_file, 'r') as src:
        # 1. Loop through all datasets in the source file
        for key in src.keys():
            if isinstance(start_index, list):
                first_half = src[key][start_index[0] : end_index[0]]
                first_half = first_half[::-1]
                second_half = src[key][start_index[1] : end_index[1]]
                sliced_data[key] = np.concatenate((first_half, second_half), axis=0)
            else:
                # 2. Read only the specified slice from the source
                sliced_data[key] = src[key][start_index : end_index]
                

    # Access internal MDTraj trajectory writer
    traj_writer = active_reporter._traj_file
    traj_writer.write(
        cell_angles = sliced_data['cell_angles'], 
        cell_lengths = sliced_data['cell_lengths'], 
        coordinates = sliced_data['coordinates'], 
        kineticEnergy = sliced_data['kineticEnergy'], 
        potentialEnergy = sliced_data['potentialEnergy'], 
        temperature = sliced_data['temperature'], 
        time = sliced_data['time'], 
        velocities = sliced_data['velocities']
    )

    traj_writer.flush()
    try:
        return (f"Successfully injected {end_index - start_index } frames into the active simulation reporter!")
    except TypeError:
        return "Done!"
    
def load_frame_with_h5py(
    simulation,
    h5_file_path: str,
    frame_index: int,
    reverse_velocity: bool = False,
    constraint_tolerance: float = 1e-7,
    write_to_reporter = False,
) -> None:
    """Load positions, velocities, and periodic box vectors from an HDF5 frame.

    Reads a single frame from an MDTraj-format HDF5 trajectory and applies
    positions, periodic box vectors, and optionally velocities to the given
    OpenMM simulation context.  Box vectors are restored *before* positions so
    that internal neighbour-list and PME grid structures are consistent with the
    new geometry, preventing NaN energies on the first force evaluation.

    Parameters
    ----------
    simulation : openmm.app.Simulation
        Active OpenMM simulation whose context will be updated in-place.
    h5_file_path : str
        Path to the MDTraj-format ``.h5`` trajectory file (read-only).
    frame_index : int
        Zero-based index of the frame to load.
    reverse_velocity : bool, optional
        If ``True``, negate all velocity components before applying them.
        Used to propagate a trajectory backward in time (default ``False``).
    constraint_tolerance : float, optional
        RMSD tolerance (dimensionless, in reduced units) passed to
        ``context.applyConstraints`` and ``context.applyVelocityConstraints``
        after position/velocity assignment (default ``1e-7``).

    Raises
    ------
    KeyError
        If the ``'coordinates'`` dataset is absent from the HDF5 file.

    Notes
    -----
    * MDTraj HDF5 convention: ``coordinates`` in nm, ``velocities`` in nm ps⁻¹,
      ``cell_lengths`` in **Ångström**, ``cell_angles`` in degrees.
    * Box vectors are constructed using the standard crystallographic → Cartesian
      conversion assuming OpenMM's lower-triangular convention.
    * ``applyConstraints`` is called after ``setPositions`` to correct any
      floating-point drift in constrained bond lengths stored in the file,
      preventing large forces on the first integration step.
    * ``applyVelocityConstraints`` is called after ``setVelocities`` to project
      out any velocity components along constrained degrees of freedom.
    """
    
    from openmm import Vec3  # local import keeps module-level namespace clean
    from mdtraj.utils import lengths_and_angles_to_box_vectors
    with h5py.File(h5_file_path, "r") as f:
        # 1. Open the H5 file and grab both datasets
        if 'cell_lengths' in f and 'cell_angles' in f:
            lengths = f['cell_lengths'][frame_index]  # Shape (3,) -> [a, b, c] in nm
            angles = f['cell_angles'][frame_index]    # Shape (3,) -> [alpha, beta, gamma] in degrees
        else:
            raise KeyError("H5 file is missing required 'cell_lengths' or 'cell_angles'.")

        # 1.2. Use MDTraj's math engine to convert lengths & angles into a 3x3 matrix of unit cell vectors
        # This handles 90° angles and skewed triclinic angles identically!
        box_matrix = lengths_and_angles_to_box_vectors(
            lengths[0], lengths[1], lengths[2],
            angles[0], angles[1], angles[2]
        )

        # 1.3. Convert the resulting NumPy matrix rows into OpenMM Vec3 objects
        a_vec = Vec3(*box_matrix[0])
        b_vec = Vec3(*box_matrix[1])
        c_vec = Vec3(*box_matrix[2])

        # 1.4. Safely apply them to the context
        simulation.context.setPeriodicBoxVectors(a_vec, b_vec, c_vec)


        # ── 2. Atomic positions ───────────────────────────────────────────────
        if "coordinates" not in f:
            raise KeyError(
                f"Dataset 'coordinates' not found in '{h5_file_path}'. "
                "Verify the file is a valid MDTraj HDF5 trajectory."
            )
        raw_positions = f["coordinates"][frame_index]           # (n_atoms, 3) nm
        simulation.context.setPositions(raw_positions * nanometers)

        # ── 3. Constraint projection for positions ────────────────────────────
        # Secondary fix: corrects sub-pm bond-length drift that would otherwise
        # produce transiently large forces on the first step.
        simulation.context.applyConstraints(constraint_tolerance)

        # ── 4. Atomic velocities ──────────────────────────────────────────────
        if "velocities" in f:
            raw_velocities = f["velocities"][frame_index]       # (n_atoms, 3) nm/ps
            if reverse_velocity:
                raw_velocities = -raw_velocities
            simulation.context.setVelocities(
                raw_velocities * (nanometers / picosecond)
            )
            # Project out velocity components along constrained bonds
            simulation.context.applyVelocityConstraints(constraint_tolerance)
            #direction = " (reversed)" if reverse_velocity else ""
            #print(f"[h5py] Frame {frame_index}: positions + velocities set{direction}.")
        #else:
            #print(f"[h5py] Frame {frame_index}: positions set; no 'velocities' dataset found in file.")

        if write_to_reporter:
            # ── 5. Write this configuration to the reporter.
            frst_reporter = simulation.reporters[0]
            copy_slice_h5_to_reporter(frst_reporter, h5_file_path, frame_index, frame_index+1)

        
                
def Create_a_transition_path_with_shooting(
    simulation,
    transition_indices: tuple,
    h5_file_path: str,
    active_h5_file: str,
    phi_atoms: list = [4, 6, 8, 14],
    psi_atoms: list = [6, 8, 14, 16],
    probability_distribution=None,
    interval: int = 10,
    max_steps: int = 5 * 10**4,
    return_nA_nB_pHipSi = False,
    scaling_factor = 1,
    shift_within_TPS = 0
) -> tuple:
    """Generate one Transition Path Sampling (TPS) shooting move.

    Implements the two-way shooting algorithm for alanine dipeptide (or any
    two-state system characterised by φ/ψ dihedral angles). A shooting point
    is drawn from the current transition path, the trajectory is propagated
    forward and/or backward in time until a stable basin is reached. The resulting 
    path is accepted or rejected via the standard TPS Metropolis criterion.

    Algorithm
    ---------
    1. **Shooting-point selection** – Draws a frame index from the provided 
       `transition_indices`. Supports both continuous paths (2 indices) and gapped 
       paths (4 indices). If `probability_distribution` is supplied, it is used as 
       the weight for selection; otherwise, selection is uniform.
    2. **Edge-frame handling** – If the shooting point coincides with the exact entry
       or exit frame of the current path, only a *single-direction* backward shot is
       attempted to minimize unnecessary computations.
    3. **Forward half-shot** – (General case only) The shooting frame is loaded with 
       original velocities and integrated forward until `angles_in_stable_region` 
       returns `True`.
    4. **Backward half-shot** – The shooting frame is reloaded with *reversed*
       velocities and integrated until `angles_in_stable_region` returns `True`
       in the *opposite* basin.
    5. **Integration Error Handling** - If an integration error occurs (e.g., `NaN` 
       energies) or if a specific coordinate constraint fails (e.g., `phi_rad > 0`), 
       the function aborts the current trajectory and recursively calls itself to 
       retry, accumulating discarded frames in `shift_within_TPS` to ensure the 
       trajectory writer alignment remains correct.
    6. **Frame Trimming & Acceptance** – The exact basin-crossing frames are located 
       within the final written intervals. Excess frames (`removed_frames`) are pruned. 
       The path is accepted with probability `min(1, L_old / L_new)`.

    Parameters
    ----------
    simulation : openmm.app.Simulation
        Active OpenMM simulation object. Its context is modified in-place during
        shooting. 
    transition_indices : tuple of int
        Tuple defining the zero-based frame indices in `h5_file_path` that make up 
        the current transition path. 
        - If 2 elements: `(entry_index, exit_index)`.
        - If 4 elements: `(entry1, exit1, entry2, exit2)` representing a gapped path.
    h5_file_path : str
        Path to the HDF5 trajectory file containing the reference transition path 
        (read-only).
    active_h5_file : str
        Path to the HDF5 trajectory file being written by the currently running
        simulation reporter. Opened in SWMR read mode to inspect newly written
        frames without interrupting the writer.
    phi_atoms : list of int, optional
        Four atom indices (0-based) defining the φ dihedral angle in the order
        [i, j, k, l]. Default is `[4, 6, 8, 14]` (alanine dipeptide in vacuum).
    psi_atoms : list of int, optional
        Four atom indices (0-based) defining the ψ dihedral angle. Default is 
        `[6, 8, 14, 16]`.
    probability_distribution : torch.Tensor or None, optional
        1-D probability tensor of length equal to the transition path. If supplied,
        `torch.multinomial` is used for selection. If `None`, uniform random 
        selection is used. Default is `None`.
    interval : int, optional
        Number of simulation steps between consecutive stability checks. Must be 
        an integer multiple of the active reporter's writing interval. Default is 10.
    max_steps : int, optional
        Hard upper bound on integration steps per half-shot. If a stable basin
        is not reached within `max_steps`, a `RuntimeError` is raised. Default is 50,000.
    return_nA_nB_pHipSi : bool, optional
        If `True`, the function alters its return signature to include basin indicator 
        statistics (N_a, N_b) and the φ/ψ values at the shooting point. Default is False.
    scaling_factor : float or int, optional
        Multiplier applied to the original transition length before calculating the 
        acceptance probability. Default is 1.
    shift_within_TPS : int, optional
        Internal accumulator for tracking the number of frames written during aborted 
        integration attempts (due to `NaN` energies or coordinate constraints). 
        Users should generally leave this at 0. Default is 0.

    Returns
    -------
    If `return_nA_nB_pHipSi` is False:
        tuple (first_half_len, backward_shot_len, removed_frames, written_len)
            - first_half_len (int or None): Number of valid frames in the 
              forward-propagated half of the new path. `None` on rejection.
            - backward_shot_len (int or None): Raw number of frames written during 
              the backward half-shot before trimming. `None` on rejection.
            - removed_frames (int or None): Number of excess frames written after the 
              detected basin-crossing frame. `None` on rejection.
            - written_len (int): Total number of frames actively written to the reporter 
              during this function call, including retries and rejected shots.
              
    If `return_nA_nB_pHipSi` is True:
        tuple (path_stats_list, N_a, N_b, dihedrals_list)
            - path_stats_list (list): `[first_half_len, backward_shot_len, removed_frames, written_len]`. 
              Values are `None` if rejected, except `written_len`.
            - N_a (int): Indicator variable representing transitions to Basin A.
            - N_b (int): Indicator variable representing transitions to Basin B.
            - dihedrals_list (list of float): `[phi_rad_shot, psi_rad_shot]` evaluated 
              exactly at the randomly selected shooting frame.

    Raises
    ------
    ValueError
        - If `interval` is not an integer multiple of the reporter's writing interval.
        - If array shapes are inconsistent when calling dihedral utilities.
    KeyError
        If the reference HDF5 file lacks a 'coordinates' dataset.
    RuntimeError
        If a stable basin is not reached within `max_steps` in either half-shot.

    Assumptions & Notes
    -------------------
    * Basin A is defined by `psi > -π/4`; basin B by `psi ≤ -π 8/9`.
    * SWMR mode must be enabled on the active HDF5 writer.
    * `load_frame_with_h5py` must handle setting positions and periodic box vectors 
      simultaneously to avoid `NaN` energies in PME simulations.
    * The acceptance criterion assumes detailed balance; non-uniform probability 
      distributions require external Rosenbluth weight corrections.
    """
    
    #Remember that the interval can be 1. Also, it is not necessary for it to be the first shot in the file. 
    writing_interval_in_steps = simulation.reporters[0]._reportInterval
    TPS_writing_steps_ratio = interval / writing_interval_in_steps
    if TPS_writing_steps_ratio%1 != 0:
        #The size of the gap will vary
        raise ValueError(
            "The interval cannot be divided into steps that are an integer multiple of the writing step size.")
    else:
        TPS_writing_steps_ratio = int(TPS_writing_steps_ratio)

    four_indices = len(transition_indices) == 4
    
    if four_indices:
        if probability_distribution is None:
            if transition_indices[0] != transition_indices[1]:
                shooting_index = np.random.choice([
                    np.random.randint(transition_indices[0], transition_indices[1]),
                    np.random.randint(transition_indices[2], transition_indices[3]),
                ])
            else:
                shooting_index = np.random.randint(transition_indices[2], transition_indices[3])
        else:
            shooting_index = torch.multinomial(probability_distribution, num_samples=1).item() + transition_indices[0]
            # probability_distribution must be the shape: transition_indices[1] - transition_indices[0] + transition_indices[2] - transition_indices[3]
            if shooting_index > transition_indices[1]:
                # Plus the gap between two parts
                #If the PROB shape is 32 and we have a path at indices (0 : 11)  (20 : 41), the path contains 32 points.
                #and if sht_idx = 11 it must be 20
                shooting_index = shooting_index + (transition_indices[2] - transition_indices[1])                               #^
                                                                                                                                #|
        original_transition_len = transition_indices[1] - transition_indices[0] +  transition_indices[3] - transition_indices[2]#| 11 - 0 + 41 -20

    else:
        if probability_distribution is None:
            shooting_index = np.random.randint(transition_indices[0], transition_indices[1])
        else:
            shooting_index = torch.multinomial(probability_distribution, num_samples=1).item() + transition_indices[0]

        original_transition_len = transition_indices[-1] - transition_indices[0] 
        original_transition_len *= scaling_factor

    if return_nA_nB_pHipSi:
        with h5py.File(h5_file_path, 'r') as f:
            try:
                coordinates = f['coordinates'][shooting_index]  # Shape: (n_atoms, 3)
            except KeyError as e:
                print(f"Error: Missing expected dataset: {e}")
        
        psi_rad_shot = numpy_dihedral(coordinates[psi_atoms[0]], coordinates[psi_atoms[1]], coordinates[psi_atoms[2]], coordinates[psi_atoms[3]])
        phi_rad_shot = numpy_dihedral(coordinates[phi_atoms[0]], coordinates[phi_atoms[1]], coordinates[phi_atoms[2]], coordinates[phi_atoms[3]])
    # ════════════════════════════════════════════════════════════════════════
    #  EDGE CASE: shooting from entry or exit frame (single-direction shot)
    # ════════════════════════════════════════════════════════════════════════
    edge_case = False
    if four_indices:
        if transition_indices[1] == transition_indices[0]:#If it is a successful edge case, the transition_indices[1] is included in the path
            edge_case = shooting_index == transition_indices[1] or shooting_index == transition_indices[3] - 1
        else:
            edge_case = shooting_index == transition_indices[1] - 1 or shooting_index == transition_indices[3] - 1
    else:
        edge_case = shooting_index == transition_indices[0] or shooting_index == transition_indices[-1] - 1 # - 1 because index -1 corresponds not to index but to slice
    if edge_case: # If we are shooting from the entry or exit frame, we want to shoot forward in time
        if not return_nA_nB_pHipSi:    
            with h5py.File(h5_file_path, 'r') as f:
                try:
                    coordinates = f['coordinates'][shooting_index]  # Shape: (n_atoms, 3)
                except KeyError as e:
                    print(f"Error: Missing expected dataset: {e}")
            
            psi_rad_shot = numpy_dihedral(coordinates[psi_atoms[0]], coordinates[psi_atoms[1]], coordinates[psi_atoms[2]], coordinates[psi_atoms[3]])

        beginning_in_A = psi_rad_shot > - np.pi/4 #if false we are in B because we started in stable region
        if return_nA_nB_pHipSi:
            N_a = 1 if beginning_in_A else 0
            
        #-------------------Revert shot----------------------------
        load_frame_with_h5py(simulation, h5_file_path, frame_index=shooting_index, reverse_velocity=True, write_to_reporter=True)
        #It is possible that we could enter and exit the stable state within the interval steps, BUT we hope this is not the case.
        converged   = False
        end_in_A    = None
        for steps in range(max_steps // interval):
            try:
                simulation.step(interval)
            except ValueError:
                shift_within_TPS += (steps+1) * TPS_writing_steps_ratio + 1
                print("Energy is NaN, one more try")
                return Create_a_transition_path_with_shooting(
                simulation,
                transition_indices,
                h5_file_path,
                active_h5_file,
                phi_atoms,
                psi_atoms,
                probability_distribution,
                interval,
                max_steps,
                return_nA_nB_pHipSi,
                scaling_factor,
                shift_within_TPS
                )
            
            # 2. Get current positions
            state = simulation.context.getState(getPositions=True)
            pos = state.getPositions(asNumpy=True).value_in_unit(nanometers)
            # Pull out coordinates directly using NumPy slicing
            phi_rad = numpy_dihedral(pos[phi_atoms[0]], pos[phi_atoms[1]], pos[phi_atoms[2]], pos[phi_atoms[3]])
            psi_rad = numpy_dihedral(pos[psi_atoms[0]], pos[psi_atoms[1]], pos[psi_atoms[2]], pos[psi_atoms[3]])
            if angles_in_stable_region(phi_rad, psi_rad):
                end_in_A = psi_rad > - np.pi/4 #if false we are in B
                converged   = True
                break

            #attempt to fix phi > 0
            if phi_rad > 0:
                print("phi_rad > 0")
                shift_within_TPS += (steps+1) * TPS_writing_steps_ratio + 1
                return Create_a_transition_path_with_shooting(
                simulation,
                transition_indices,
                h5_file_path,
                active_h5_file,
                phi_atoms,
                psi_atoms,
                probability_distribution,
                interval,
                max_steps,
                return_nA_nB_pHipSi,
                scaling_factor,
                shift_within_TPS
                )
        written_len = (steps+1) * TPS_writing_steps_ratio + 1 + shift_within_TPS
        if not converged:
            raise RuntimeError(
                f"Edge-frame backward shot did not reach a stable basin within "
                f"{max_steps} steps.  Increase max_steps or check basin definitions."
            )
        
        if end_in_A == beginning_in_A:
            if return_nA_nB_pHipSi: #This is safe because, if we return nA_nB, we return [...], ... before ''return None, None, None, int(written_len)''
                if end_in_A:
                    N_a += 1
                return [None, None, None, int(written_len)], N_a, 2 - N_a, [phi_rad_shot, psi_rad_shot]

            print("Non reactive")
            return None, None, None, int(written_len)
        else:
            #Check if EXACTLY ONE frame was in a stable state in the last ten frames
            #1. Open the file in safe read-only SWMR mode
            if TPS_writing_steps_ratio == 1: 
            #If there is no gap, there is no need for heavy calculation.
                removed_frames = 0
            else:
                with h5py.File(active_h5_file, 'r', swmr=True) as f:
                    removed_frames = _find_crossing_from_dataset(
                    f["coordinates"], TPS_writing_steps_ratio, phi_atoms, psi_atoms
                    )
                

            #6. ACCEPTANCE CRITERIA
            new_path_len = written_len - removed_frames
            if new_path_len == 0:
                print("SOMETHING HAPPENED")
            if min(1,  original_transition_len / new_path_len) > np.random.rand():
                if return_nA_nB_pHipSi: #This is safe because, if we return nA_nB, we return [...], ... before ''return None, None, None, int(written_len)''
                    return [0, int(written_len), int(removed_frames), int(written_len)], 1, 1, [phi_rad_shot, psi_rad_shot]
                print("success")
                return  0, int(written_len), int(removed_frames), int(written_len)
            else:
                if return_nA_nB_pHipSi:
                    return [None, None, None, int(written_len)], 1, 1, [phi_rad_shot, psi_rad_shot]
                return None, None, None, int(written_len)
            
            #Output: 2 indices of the end of first half of the constructed shot and the end of second
    
    # ════════════════════════════════════════════════════════════════════════
    #  GENERAL CASE: interior shooting point (two-way shot)
    # ════════════════════════════════════════════════════════════════════════
    # ── FORWARD HALF-SHOT ─────────────────────────────────────────────────
    #1. Set the simulation to the shooting frame
    load_frame_with_h5py(simulation, h5_file_path, frame_index=shooting_index)
    #It is possible that we could enter and exit the stable state within the interval steps, BUT we hope this is not the case.
    converged    = False
    beginning_in_A = None
    for steps in range(max_steps // interval):
        #2. Simulate backward in time 

        try:
            simulation.step(interval)
        except ValueError:
            shift_within_TPS += (steps+1) * TPS_writing_steps_ratio
            print("Energy NaN, one more try")
            return Create_a_transition_path_with_shooting(
            simulation,
            transition_indices,
            h5_file_path,
            active_h5_file,
            phi_atoms,
            psi_atoms,
            probability_distribution,
            interval,
            max_steps,
            return_nA_nB_pHipSi,
            scaling_factor,
            shift_within_TPS
            )
       

        # 3. Get current positions
        state = simulation.context.getState(getPositions=True)
        pos = state.getPositions(asNumpy=True).value_in_unit(nanometers)
        #4. Pull out coordinates directly using NumPy slicing, which more efficient then creating a dummy MDTraj trajectory
        phi_rad = numpy_dihedral(pos[phi_atoms[0]], pos[phi_atoms[1]], pos[phi_atoms[2]], pos[phi_atoms[3]])
        psi_rad = numpy_dihedral(pos[psi_atoms[0]], pos[psi_atoms[1]], pos[psi_atoms[2]], pos[psi_atoms[3]])
        if angles_in_stable_region(phi_rad, psi_rad):
            beginning_in_A = psi_rad > - np.pi/4 #if false we are in B
            if return_nA_nB_pHipSi:
                if beginning_in_A:
                    N_a = 1
                else:
                    N_a = 0
            converged = True
            break

        #attempt to fix phi > 0
        if phi_rad > 0:
            print("phi_rad > 0")
            shift_within_TPS += (steps+1) * TPS_writing_steps_ratio
            return Create_a_transition_path_with_shooting(
            simulation,
            transition_indices,
            h5_file_path,
            active_h5_file,
            phi_atoms,
            psi_atoms,
            probability_distribution,
            interval,
            max_steps,
            return_nA_nB_pHipSi,
            scaling_factor,
            shift_within_TPS
            )
    #5. Clearing the rest frames in the last interval to make sure we are capturing onlu one stable state in the last interval
    forward_shot_len = (steps + 1) * TPS_writing_steps_ratio #The number of frames WRITTEN for forward shooting
    if not converged:
        raise RuntimeError(
            f"Forward half-shot did not reach a stable basin within "
            f"{max_steps} steps.  Increase max_steps or check basin definitions."
        )
    #NOTE  #The last index in the simulation file is the starting index of the path
    # ── Open active file for both crossing checks ───────────────────────
    if TPS_writing_steps_ratio == 1: 
    #If there is no gap, there is no need for heavy calculation.
        fwd_removed = 0
    else:
        with h5py.File(active_h5_file, "r", swmr=True) as f:
            coord_dataset = f["coordinates"]
            fwd_removed = _find_crossing_from_dataset(
                coord_dataset, TPS_writing_steps_ratio, phi_atoms, psi_atoms
            )

    # ── BACKWARD HALF-SHOT ────────────────────────────────────────────────
    #1. Set the simulation to the shooting frame
    #1.1 Save the shooting configuration to the backward half-shot
    load_frame_with_h5py(simulation, h5_file_path, frame_index=shooting_index, reverse_velocity=True, write_to_reporter=True)
    #It is possible that we could enter and exit the stable state within the interval steps, BUT we hope this is not the case.
    converged = False
    end_in_A  = None
    for steps in range(max_steps // interval):
        #2. Simulate forward in time 
        try:
            simulation.step(interval)
        
        except ValueError:
            #here we go again
            print("Energy NaN, one more try")
            shift_within_TPS += (steps+1) * TPS_writing_steps_ratio + 1 + forward_shot_len
            return Create_a_transition_path_with_shooting(
            simulation,
            transition_indices,
            h5_file_path,
            active_h5_file,
            phi_atoms,
            psi_atoms,
            probability_distribution,
            interval,
            max_steps,
            return_nA_nB_pHipSi,
            scaling_factor,
            shift_within_TPS
            )
        
        # 3. Get current positions
        state = simulation.context.getState(getPositions=True)
        pos = state.getPositions(asNumpy=True).value_in_unit(nanometers)
        #4. Pull out coordinates directly using NumPy slicing, which more efficient then creating a dummy MDTraj trajectory
        phi_rad = numpy_dihedral(pos[phi_atoms[0]], pos[phi_atoms[1]], pos[phi_atoms[2]], pos[phi_atoms[3]])
        psi_rad = numpy_dihedral(pos[psi_atoms[0]], pos[psi_atoms[1]], pos[psi_atoms[2]], pos[psi_atoms[3]])
        if angles_in_stable_region(phi_rad, psi_rad):
            end_in_A = psi_rad > - np.pi/4 #if false we are in B
            if return_nA_nB_pHipSi:
                if end_in_A:
                    N_a += 1
            converged = True
            break

        #attempt to fix phi > 0
        if phi_rad > 0:
            print("phi_rad > 0")
            shift_within_TPS += (steps+1) * TPS_writing_steps_ratio + 1 + forward_shot_len
            return Create_a_transition_path_with_shooting(
            simulation,
            transition_indices,
            h5_file_path,
            active_h5_file,
            phi_atoms,
            psi_atoms,
            probability_distribution,
            interval,
            max_steps,
            return_nA_nB_pHipSi,
            scaling_factor,
            shift_within_TPS
            )
    #5. Check if EXACTLY ONE frame was in a stable state in the last ten frames
    backward_shot_len = (steps + 1) * TPS_writing_steps_ratio + 1 #The number of frames WRITTEN for reverse shooting
    # "+ 1" because of the manually saved shooting configuration.
    if not converged:
        raise RuntimeError(
            f"Backward half-shot did not reach a stable basin within "
            f"{max_steps} steps.  Increase max_steps or check basin definitions."
        )
    
    written_len = forward_shot_len + backward_shot_len + shift_within_TPS
    # Same-basin shot → reject before expensive file I/O
    if end_in_A == beginning_in_A:
        
        if return_nA_nB_pHipSi:#This is safe because, if we return nA_nB, we return [...], ... before ''return None, None, None, int(written_len)''
            return [None, None, None, int(written_len)], N_a, 2 - N_a, [phi_rad_shot, psi_rad_shot]
        return None, None, None, int(written_len)  
    

    if TPS_writing_steps_ratio == 1: 
    #If there is no gap, there is no need for heavy calculation.
        removed_frames = 0
    else:
    # ── Open active file ONCE for both crossing checks ───────────────────────
    # The forward crossing was already written before the backward half-shot
    # started.  We split the single open across the two reads using seek:
    #   • forward crossing  → coord_dataset[-written_len : -backward_shot_len]
    #     — but we only need the last ``interval`` frames of the forward half,
    #       which are at positions [-written_len : -written_len + interval]
    #     — equivalently: [-backward_shot_len - interval : -backward_shot_len]
    #   • backward crossing → coord_dataset[-interval:]
    #5.1. Open the file in safe read-only SWMR mode
    
        with h5py.File(active_h5_file, "r", swmr=True) as f:
            coord_dataset = f["coordinates"]
            # Backward crossing: last ``interval`` frames of the whole written block
            removed_frames = _find_crossing_from_dataset(
                coord_dataset, TPS_writing_steps_ratio, phi_atoms, psi_atoms
            )
    second_half_len = backward_shot_len - removed_frames
    first_half_len  = forward_shot_len  - fwd_removed

    #-------------------Acceptance criteria----------------------------
    new_path_len = first_half_len + second_half_len
    if min(1,  original_transition_len / new_path_len) > np.random.rand():
        #print("success")
        if return_nA_nB_pHipSi:
            return [int(first_half_len), int(backward_shot_len), int(removed_frames), int(written_len)], 1, 1, [phi_rad_shot, psi_rad_shot]
        
        return int(first_half_len), int(backward_shot_len), int(removed_frames), int(written_len)
    else:
        if return_nA_nB_pHipSi:
            return [None, None, None, int(written_len)], 1, 1, [phi_rad_shot, psi_rad_shot]
        return None, None, None, int(written_len)

# Machine Learning Part

In [ ]:
# ── 1. Neural Network ────────────────────────────────────────────────────────

class CommittorNet(nn.Module):
    """
    Maps a configuration q (dim-dimensional coordinate vector)
    to a scalar N(q).  The committor is then sigmoid(N(q)).

    Architecture: input → Linear(hidden_dims) + SiLU → Linear(1)
    SiLU (Swish) is chosen as the smooth activation function:
        SiLU(x) = x · sigmoid(x)
    It is everywhere differentiable, non-saturating for positive inputs,
    and empirically outperforms tanh/sigmoid in deep networks.
    """

    def __init__(self, input_dim: int, hidden_dims: list[int] = None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128] * 6
        
        layers = []
        in_dim = input_dim
        for n_hidden in hidden_dims:
            layers.append(nn.Linear(in_dim, n_hidden))
            layers.append(nn.SiLU())          # smooth activation without sigmoid problems (vanishing signal)
            in_dim = n_hidden

        layers.append(nn.Linear(in_dim, 1))   # scalar output N(q)

        self.net = nn.Sequential(*layers)

        # Weight initialisation: Xavier uniform keeps gradients well-scaled
        # through the deep stack
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, q: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        q : (batch, input_dim)

        Returns
        -------
        N_q : (batch,)   raw scalar output N(q)
        """
        return self.net(q).squeeze(-1)

    def save_weights(self, file_name: str = 'model_weights'):
        torch.save(self.state_dict(), file_name+'.pth')

    def load_weights(self, file_name: str = 'model_weights.pth'):
        self.load_state_dict(torch.load(file_name, weights_only=True))


# ── 2. Committor function ────────────────────────────────────────────────────

def committor(N_q: torch.Tensor) -> torch.Tensor:
    """
    P_B(q) = sigmoid( N(q) ) = 1 / (1 + exp(-N(q)))

    Guaranteed to lie in (0, 1).
    """
    return torch.sigmoid(N_q)


# ── 3. Selection probability ─────────────────────────────────────────────────

def selection_probability(
    N_q: torch.Tensor,
    lam: float = 1.0,
) -> torch.Tensor:
    """
    P_sel(q_s | X) = 1 / sum_{q_i in X} [ (N(q_s)^2 + λ^2) /
                                            (N(q_i)^2 + λ^2) ]

    Parameters
    ----------
    N_q_s   : (M,) –  N(q) for all configurations in the path X
    lam   : regularisation parameter λ

    Returns
    -------
    P_sel : same shape as N_q_s
    """
    lam2   = lam ** 2
    N_mod = N_q ** 2 + lam2
    normalisation = (1/N_mod).sum()
    return 1 / (normalisation * N_mod)


# ── 4. Expected number of transition paths ───────────────────────────────────

def n_tps_expected(P_B: torch.Tensor) -> torch.Tensor:
    """
    n_TPS_exp = Σ_i  2 · P_B(q_i) · (1 − P_B(q_i))

    Parameters
    ----------
    P_B : (k,)  committor values at k shooting points

    Returns
    -------
    scalar tensor
    """
    return 2.0 * (P_B * (1.0 - P_B)).sum()


# ── 5. Efficiency coefficient ────────────────────────────────────────────────

def efficiency_coefficient(
    n_tps_gen: float | torch.Tensor,
    n_tps_exp: torch.Tensor,
) -> torch.Tensor:
    """
    α_eff = min( 1,  (1 − n_TPS_gen / n_TPS_exp)^2 )

    Parameters
    ----------
    n_tps_gen : number of transition paths actually generated
    n_tps_exp : expected number (output of n_tps_expected)

    Returns
    -------
    scalar tensor in [0, 1]
    """
    ratio = n_tps_gen / n_tps_exp
    alpha = (1.0 - ratio) ** 2
    return torch.clamp(alpha, max=1.0)


# ── 6. Binomial negative log-likelihood loss ─────────────────────────────────

def binomial_nll_loss(
    P_B: torch.Tensor,
    n_A: torch.Tensor,
    n_B: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Negative log-likelihood of the binomial shooting outcome.

    −ln ∏_i p(n_A_i, n_B_i | q_i)

    where  p(n_A, n_B | q) = C(n_A+n_B, n_B)
                              · (1−P_B)^n_A · P_B^n_B

    The log-binomial-coefficient is computed via the log-gamma function
    (stable for non-integer counts too):
        ln C(n,k) = ln Γ(n+1) − ln Γ(k+1) − ln Γ(n−k+1)

    Parameters
    ----------
    P_B : (k,)   committor at each shooting point
    n_A : (k,)   number of A-terminating trajectories  (float or int tensor)
    n_B : (k,)   number of B-terminating trajectories

    Returns
    -------
    scalar loss
    """
    n_A  = n_A.float()
    n_B  = n_B.float()
    n    = n_A + n_B

    # log C(n, n_B)  –  numerically stable via log-gamma
    log_binom = gammaln(n + 1) - gammaln(n_B + 1) - gammaln(n_A + 1)

    # log-likelihood per shooting point
    log_lik = (
        log_binom
        + n_A * torch.log(1.0 - P_B + eps)
        + n_B * torch.log(P_B + eps)
    )

    return -log_lik.sum()


# ── 7. Training loop ─────────────────────────────────────────────────────────

def train_step(
    model:     CommittorNet,
    q_configs: torch.Tensor,      # (K, input_dim)  – all K shooting points
    n_A:       torch.Tensor,      # (K,)
    n_B:       torch.Tensor,      # (K,)
    n_tps_gen: int,               # actual reactive paths counted this iteration
    optimizer: optim.Optimizer,
    scheduler: optim.lr_scheduler._LRScheduler = None,
    alpha_eff_threshold: float = 0.3,
    device:    torch.device = torch.device("cpu"),
) -> dict:
    """
    One learning iteration.

    Steps
    -----
    1.  Forward pass → N(q), P_B(q)
    2.  Compute α_eff; skip backprop if α_eff < threshold
    3.  Backprop on binomial NLL loss

    Returns
    -------
    dict with keys: loss, alpha_eff, n_tps_exp, P_B
    """
    
    model.train()                               # set model to training mode
    q_configs = q_configs.to(device)
    n_A       = n_A.to(device)
    n_B       = n_B.to(device)
    
    #print(q_configs.shape, n_A.shape, n_B.shape)

    N_q = model(q_configs)                      # (K,)
    P_B = committor(N_q)                        # (K,)

    n_exp = n_tps_expected(P_B)
    alpha = efficiency_coefficient(n_tps_gen, n_exp)

    result = {
        "alpha_eff": alpha.item(),
        "n_tps_exp": n_exp.item(),
        "P_B":       P_B.detach().cpu()
    }

    if alpha.item() < alpha_eff_threshold:
        print(f"  α_eff = {alpha.item():.4f} < {alpha_eff_threshold} → skipping update")
        return result

    loss = binomial_nll_loss(P_B, n_A, n_B)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if not (scheduler is  None):
        scheduler.step(loss)

    return result

# ── 8. Evaluation function ──────────────────────────────────────────────────

def evaluate_model(model: CommittorNet, committor_grid: np.ndarray = None, print_min_max: bool = False, return_min_max: bool = False):
    """Evaluate a neural network committor model over a 2D spatial grid and visualize predictions or residuals.

    This function passes a 2D meshgrid of coordinates through a trained `CommittorNet` 
    model to compute raw model outputs $N(q)$, applies a sigmoid transformation to map 
    outputs to committor probabilities $P_B = \frac{1}{1 + e^{-N(q)}}$, and renders a 2D 
    heatmap of the results. 

    Depending on whether a reference `committor_grid` is supplied, the function operates 
    in one of two modes:
    1. **Standalone Prediction Mode (`committor_grid=None`):** Evaluates the model on an 
       $80 \times 80$ grid over the domain $[-\pi, 0] \times [-\pi, 0]$ (typically 
       representing $\phi / \psi$ dihedral space) and plots $P_B$.
    2. **Reference Comparison Mode (`committor_grid` provided):** Evaluates the model on 
       a grid over $[-3, 3] \times [-3, 3]$ matching the dimensions of `committor_grid`, 
       computes the residual difference ($P_{B, \text{model}} - P_{B, \text{ref}}$), and 
       plots a difference heatmap.

    Parameters
    ----------
    model : CommittorNet
        A PyTorch neural network predicting unnormalized committor values $N(q)$ 
        for 2D input coordinates $q = (\phi, \psi)$.
    committor_grid : np.ndarray or None, optional
        A 2D array of reference committor probabilities $P_B \in [0, 1]$. If provided, 
        the function computes and visualizes the difference between model predictions 
        and this reference grid. Default is `None`.
    print_min_max : bool, optional
        If `True` and `committor_grid` is provided, prints the minimum and maximum 
        residual differences between predicted and reference $P_B$ values. Default is `False`.
    return_min_max : bool, optional
        If `True` and `committor_grid` is provided, returns the minimum and maximum 
        residual differences as a tuple. Default is `False`.

    Returns
    -------
    tuple of (float, float) or None
        - If `return_min_max=True` and `committor_grid` is provided: Returns 
          `(min_diff, max_diff)`, ignoring `NaN` values.
        - Otherwise: Returns `None`.

    Raises
    ------
    ValueError
        If `committor_grid` is supplied but its shape does not match the 2D meshgrid 
        generated from its dimensions.

    Notes
    -----
    * **Device Handling:** Inputs are automatically moved to the device of the model's 
      parameters (CPU or GPU).
    * **Zero Difference Handling:** In comparison mode, exact zero differences 
      ($P_{B, \text{model}} - P_{B, \text{ref}} = 0$) are masked as `np.nan` prior to 
      visualization and min/max calculation to highlight true variance.
    * **Colormaps:** Standalone predictions are plotted using the `'magma'` colormap 
      over $[0, 1]$, whereas residual differences are plotted using the divergent `'brg'` 
      colormap over $[-1, 1]$.
    """
    model.eval()
    with torch.no_grad():
        if committor_grid is None:
            x = torch.linspace(-np.pi, 0, 80)
            y = torch.linspace(-np.pi, 0, 80)
        else:
            x = torch.linspace(-3, 3, committor_grid.shape[0])
            y = torch.linspace(-3, 3, committor_grid.shape[1])
        X, Y = torch.meshgrid(x, y, indexing='ij') # create a grid of points in the configuration space
        # pass the grid through the model to get N(q) values, then reshape back to grid form
        q = torch.stack((X.flatten(), Y.flatten()), dim=-1).to(next(model.parameters()).device) 
        N_q = model(q).cpu().numpy().reshape(X.shape)
        P_b = 1/(1 + np.exp(-N_q))
        if committor_grid is None:
            values = P_b .T  # (y_tiles, x_tiles)

            plt.figure(figsize=(8, 6))

            im = plt.pcolormesh(x, y, values, cmap='magma', vmin=0, vmax=1, shading='auto')
            plt.colorbar(im, label='Committor probability')
            plt.xlabel('$\phi$')
            plt.ylabel('$\psi $')
            plt.title('Committor probability')
            plt.tight_layout()
            plt.show()
            if return_min_max or print_min_max:
                print("There is nothing to compare the model with.")
        else:
            if committor_grid.shape != P_b.shape:
                raise ValueError("Provided committor_grid shape does not match model output shape.")
            diff = P_b - committor_grid
            diff = np.where(diff == 0, np.nan, diff)  # Set zero differences to NaN for better visualization
            if print_min_max:
                print(f"The maximal difference between the model and the reference committor is {np.nanmax(diff)}")
                print(f"The minimal difference between the model and the reference committor is {np.nanmin(diff)}")
            if return_min_max:
                return np.nanmin(diff), np.nanmax(diff)
            plt.figure(figsize=(8, 6))
            im = plt.pcolormesh(x, y, diff.T, cmap='brg', vmin=-1, vmax=1, shading='auto')
            plt.colorbar(im, label='Model P_B - Reference P_B')
            plt.xlabel('$\phi$')
            plt.ylabel('$\psi $')
            plt.title('Difference between Model and Reference Committor')
            plt.tight_layout()
            plt.show()
            
            
def plot_isocommittor_line(committor_grid: np.ndarray, num_lines: int = 10, alpha_of_simul_committor = 50, plot_stable_states = True):
    """Visualize isocommittor lines and stable states on a 2D probability grid.

    This function generates a contour plot of a provided 2D committor probability grid 
    over the predefined spatial domain $[-\pi, 0] \times [-\pi, 0]$ (representing 
    $\phi$ and $\psi$ dihedral angles). It draws evenly spaced isocommittor lines 
    (contours of constant probability) and specifically highlights the boundary states: 
    State A ($P_B \approx 0$) in blue, and State B ($P_B \approx 1$) in red.

    Additionally, it can optionally overlay a scatter plot of the exact stable regions 
    by evaluating an external `angles_in_stable_region` mask over the grid.

    Parameters
    ----------
    committor_grid : np.ndarray
        A 2D array of committor probabilities, typically bounded between 0 and 1. The 
        shape of this array dictates the resolution of the generated spatial meshgrid.
    num_lines : int, optional
        The number of evenly spaced contour levels to draw between 0 and 1 (inclusive). 
        Default is 10.
    alpha_of_simul_committor : int or float, optional
        Currently unused in the function body. Maintained for signature compatibility. 
        Default is 50.
    plot_stable_states : bool, optional
        If True, evaluates the external `angles_in_stable_region` function over the 
        meshgrid and overlays cyan scatter points to explicitly denote the stable 
        basins. Default is True.

    Returns
    -------
    None
        The function does not return any values; it directly renders and displays 
        the matplotlib figure.

    Notes
    -----
    * **Spatial Domain:** The grid coordinates are strictly hardcoded to span from 
      $-\pi$ to $0$ for both the x-axis ($\phi$) and y-axis ($\psi$). 
    * **Dependencies:** This function relies on `torch.linspace` to generate the 
      coordinate vectors before passing them to `np.meshgrid`. It also requires the 
      external `angles_in_stable_region` callable to be defined in the global scope 
      if `plot_stable_states=True`.
    * **Contour Highlighting:** Boundary states are detected using `np.isclose` with 
      default tolerances against 0 and 1.
    """
    x = torch.linspace(-np.pi, 0, committor_grid.shape[0])
    y = torch.linspace(-np.pi, 0, committor_grid.shape[1])
    X, Y = np.meshgrid(x, y, indexing='ij')
    plt.figure(figsize=(8, 6))
    points = np.stack((X, Y), axis=-1)  # shape (len(y), len(x), 2)

    levels = np.linspace(0, 1, num_lines) 
    contour = plt.contour(X, Y, committor_grid, levels=levels, colors='green')
    plt.clabel(contour, inline=True, fontsize=8)
    plt.contour(X, Y, np.isclose(committor_grid, 1), colors='red', linewidths=2)  # Highlight the stable state B
    plt.contour(X, Y, np.isclose(committor_grid, 0), colors='blue', linewidths=2)  # Highlight the stable state A

    if plot_stable_states:
        stable_mask = angles_in_stable_region(points[..., 0], points[..., 1])
        plt.scatter(X[stable_mask], Y[stable_mask], color='cyan', label='Stable Region ', s=5)
    plt.xlabel('$\phi$')
    plt.ylabel('$\psi $')
    plt.title('Isocommittor Lines')
    plt.tight_layout()
    plt.show()

def plot_confusion_plot(shooting_points, n_a, model: torch.nn.Module, threshold: float = 0.05):
    """Generate a confusion/fidelity plot comparing neural network committor predictions against empirical TPS shooting outcomes.

    This function evaluates a trained PyTorch committor model on dynamic Transition Path Sampling 
    (TPS) shooting points rather than a uniform grid. It converts empirical outcome counts $n_A$ 
    (trajectories ending in Basin A) into estimated committor probabilities 
    $P_B = \frac{2 - n_A}{2}$, transforms the model's raw outputs using an external `committor()` 
    function, and plots predicted versus empirical values sorted by increasing empirical $P_B$.

    Parameters
    ----------
    shooting_points : np.ndarray
        Array of shooting point coordinates with shape `(num_epochs, K, 2)` or `(N, 2)` 
        containing 2D spatial coordinates $(\phi, \psi)$ in radians.
    n_a : np.ndarray
        Array of shape `(num_epochs, K)` or `(N,)` containing the integer count of trajectories 
        per shooting move that terminated in Basin A (typically $n_A \in \{0, 1, 2\}$).
    model : torch.nn.Module
        A PyTorch neural network that accepts 2D coordinate inputs $(\phi, \psi)$ and 
        outputs raw unnormalized committor network values $N(q)$.
    threshold : float, optional
        The half-width of the error band plotted around the empirical committor values. 
        Default is 0.05.

    Returns
    -------
    None
        The function does not return any values; it directly renders and displays 
        the matplotlib figure.

    Notes
    -----
    * **Committor Mapping:** The empirical committor $P_B$ is computed as $P_B = \frac{2 - n_A}{2}$:
      - $n_A = 2 \implies P_B = 0.0$ (both shots hit Basin A)
      - $n_A = 1 \implies P_B = 0.5$ (one shot hit A, one hit B; transition region)
      - $n_A = 0 \implies P_B = 1.0$ (both shots hit Basin B)
    * **Dependencies:** Relies on an external `committor(N)` callable in the global scope 
      to map raw model outputs to probabilities in $[0, 1]$.
    * **Device Compatibility:** Tensors are automatically sent to the device (CPU or GPU) 
      where `model` parameters reside.
    * **Visualization:** Unlike standard grid plots, shooting points concentrate naturally in the 
      transition region ($P_B \approx 0.5$). Sorting points by empirical $P_B$ allows rapid 
      visual inspection of systematic over- or under-prediction across the sampled dataset.
    """
    shooting_points_flatten = shooting_points.reshape(-1, 2)  # shape (num_epochs * K, 2)
    n_a_flatten = n_a.flatten()  # shape (num_epochs * K,)
    committor_array = (2 - n_a_flatten) / 2  # Convert n_a to committor values (0, 1, or 2)
    device = next(model.parameters()).device 
    with torch.no_grad():
        predicted_N_device = model(torch.tensor(shooting_points_flatten, dtype=torch.float32).to(device))
        predicted_P_B_device = committor(predicted_N_device)
        predicted_P_B = predicted_P_B_device.cpu().numpy() #shape (num_epochs * K,)
    
    sorted_indices = np.argsort(committor_array)
    committor_array = committor_array[sorted_indices]
    predicted_P_B = predicted_P_B[sorted_indices]

    x = np.arange(len(predicted_P_B))  # Index for each point in the grid
    
    #Since we have no grid, two shooting points that are close together can be calculated separately.
    plt.figure(figsize=(12, 9))
    plt.fill_between(x, committor_array - threshold, committor_array + threshold, color='green', alpha=0.2, label='±1 threshold')
    plt.scatter(x, predicted_P_B, color='blue', label='Predicted P_B')
    
    plt.xlabel('Order of mesh grid point')
    plt.ylabel('Committor Probability')
    plt.title(f'Confusion Plot at allowed error {threshold}')
    plt.legend()
    plt.tight_layout()
    plt.show()

"""def plot_confusion_plot_2(committor_grid, shooting_points, model, threshold = 0.05, show_diviation_plot=False):
    "
    Evaluate and plot committor model predictions at shooting point locations using spatial grid-averaging.

    This function evaluates a PyTorch neural network committor model on a set of 2D shooting points 
    $(\phi, \psi)$, converts predictions to probabilities via an external `committor()` function, 
    and spatially bins the predictions into a 2D grid matching the shape of `committor_grid`. 
    By computing the weighted average of predicted committors within each spatial bin 
    (`H_weights / H_counts`), it enables direct cell-by-cell comparison against the ground-truth 
    `committor_grid`.

    Depending on `show_diviation_plot`, the function renders one of two visualizations:
    1. **Fidelity Plot (`show_diviation_plot=False`):** Flattens and sorts grid cells by 
       ascending reference committor probability. Plots spatially binned model predictions 
       colored by whether they fall within `±threshold` of the ground-truth value.
    2. **Deviation Plot (`show_diviation_plot=True`):** Plots the raw numerical residuals 
       ($P_{B, \text{true}} - P_{B, \text{predicted}}$) across non-empty grid cells alongside 
       dashed horizontal threshold markers at `±threshold`.

    Parameters
    ----------
    committor_grid : np.ndarray
        A square 2D array (`bins` x `bins`) containing reference ground-truth committor probabilities. 
        Its dimension `shape[0]` determines the number of spatial histogram bins.
    shooting_points : np.ndarray
        Array of shooting point coordinates of shape `(num_epochs, K, 2)` or `(N, 2)` 
        containing 2D spatial coordinates $(\phi, \psi)$ in radians.
    model : torch.nn.Module
        A PyTorch neural network that accepts 2D coordinate inputs $(\phi, \psi)$ and 
        outputs raw unnormalized committor values $N(q)$.
    threshold : float, optional
        The maximum tolerable absolute error between the reference committor and the 
        binned model prediction. Default is 0.05.
    show_diviation_plot : bool, optional
        If `True`, displays a residual scatter plot of deviation values instead of the 
        sorted committor probability comparison. Default is `False`.

    Returns
    -------
    None
        The function does not return any values; it directly renders and displays 
        the matplotlib figure.

    Notes
    -----
    * **Spatial Histogram Averaging:** Grid cells containing no shooting points yield 0 counts, 
      resulting in `NaN` during division (`H_weights / H_counts`). These unvisited cells 
      are automatically filtered out in the deviation plot and placed at the sorted indices 
      in the standard plot.
    * **Dependencies:** Relies on an external `committor(N)` callable in the global scope 
      to map raw model outputs to probabilities in $[0, 1]$.
    * **Device Handling:** Input coordinates are automatically sent to the device (CPU or GPU) 
      where the model parameters reside.
    "
    shooting_points_flatten = shooting_points.reshape(-1, 2)# shape (num_epochs * K, 2)
    
    device = next(model.parameters()).device 
    with torch.no_grad():
        predicted_N_device = model(torch.tensor(shooting_points_flatten, dtype=torch.float32).to(device))
        predicted_P_B_device = committor(predicted_N_device)
        predicted_P_B = predicted_P_B_device.cpu().numpy() #shape (num_epochs * K,)
    
    phi = shooting_points_flatten[:, 0] # shape (num_epochs * K,)
    psi = shooting_points_flatten[:, 1] # shape (num_epochs * K,)
    
    bins = committor_grid.shape[0] # committor_grid is a square grid
    H_weights, phiedges, psiedges = np.histogram2d(phi, psi, bins=bins, weights=predicted_P_B) 
    H_counts, _, _ = np.histogram2d(phi, psi, bins=bins)
    H_predicted = H_weights / (H_counts) 

    plt.figure(figsize=(12, 9))
    if show_diviation_plot:
        H_deviation = committor_grid - H_predicted
        deviation = H_deviation.ravel() # or H.flatten()
        deviation = deviation[~np.isnan(deviation)]
        plt.plot(deviation, 'x')
        plt.axhline(threshold, color='red', linestyle='--', label='Threshold')
        plt.axhline(-threshold, color='red', linestyle='--')
        plt.xlabel('Index of grid point')
        plt.ylabel('Deviation from true committor')
        plt.title(f'Confusion Plot with grid averaging at allowed error {threshold}')
        plt.legend()
    else:
        committor_array = committor_grid.ravel()
        model_array = H_predicted.ravel()
        sorted_indices = np.argsort(committor_array)
        committor_array = committor_array[sorted_indices]
        model_array = model_array[sorted_indices]
        TP_mask = (np.abs(model_array - committor_array) <= threshold)
        FP_mask = ~TP_mask
        x = np.arange(len(committor_array))
        plt.scatter(x[TP_mask], model_array[TP_mask], c = 'green', marker='o', label='True Predicted Committor')
        plt.scatter(x[FP_mask], model_array[FP_mask], c = 'red', marker='x', label='False Predicted Committor')
        plt.fill_between(np.arange(len(committor_array)), committor_array - threshold, committor_array + threshold, color='green', alpha=0.2, label='±1 threshold')
        plt.xlabel('Index of grid point')
        plt.ylabel('Committor Probability')
        plt.title(f'Comparison of True and Predicted Committor')
        plt.legend()
    plt.tight_layout()
    plt.show()
"""

def plot_shooting_point_density(shooting_points : np.ndarray):
    """Scatter plot Transition Path Sampling (TPS) shooting points in φ/ψ dihedral space colored by simulation epoch.

    This function visualizes the spatial distribution and temporal progression of TPS shooting 
    points across simulation epochs. The input array is flattened into 2D coordinate pairs 
    $(\phi, \psi)$ and plotted as a semi-transparent scatter plot using the `'cool'` colormap, 
    allowing visual identification of how shooting point selection drifts or concentrates 
    over the course of sampling.

    Parameters
    ----------
    shooting_points : np.ndarray
        A 3D NumPy array of shape `(num_epochs, K, 2)` containing shooting point coordinates:
        - `num_epochs`: The number of TPS iterations or sampling epochs.
        - `K`: The number of shooting points recorded per epoch.
        - `2`: The $(\phi, \psi)$ dihedral angle coordinates (in radians) at indices 0 and 1, respectively.

    Returns
    -------
    None
        The function does not return any values; it directly renders and displays 
        the matplotlib figure.

    Notes
    -----
    * **Color Mapping:** Points are colored according to their epoch index normalized between 1 and num_epochs. 
      The accompanying colorbar is rescaled to display integer epoch numbers ranging from `1` to `num_epochs`.
    * **Plot Styling:** Points are rendered with a marker size of `s=5` and transparency `alpha=0.5` 
      to clearly highlight density clusters and overlapping points across multiple epochs.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = np.linspace(0, 1, shooting_points.shape[0])
    colors = np.tile(colors[:, np.newaxis], (1, shooting_points.shape[1]))# shape (num_epochs, K)
    shooting_points_flatten = shooting_points.reshape(-1, 2)
    colors = colors.flatten()  # shape (num_epochs * K,)
    phi = shooting_points_flatten[:, 0] # shape (K, 2)
    psi = shooting_points_flatten[:, 1] # shape (K, 2)
    plt.scatter(phi, psi, c=colors, cmap='cool', s=5, alpha=0.5)
    sm = plt.cm.ScalarMappable(cmap='cool', norm=plt.Normalize(vmin=1, vmax=shooting_points.shape[0]))
    plt.colorbar(sm, ax=ax, label='Epoch')
    ax.set_xlabel('$\phi$')
    ax.set_ylabel('$\psi $')
    ax.set_title('Shooting Point Density')
    plt.tight_layout()
    plt.show()

def plot_raugh_commitor(shooting_points, n_a, bins = 100, return_grid = False):
    fig, ax = plt.subplots(figsize=(8, 6))
    shooting_points_flatten = shooting_points.reshape(-1, 2)  # shape (num_epochs * K, 2)
    n_a_flatten = n_a.flatten()  # shape (num_epochs * K,)
    weights = 2 - n_a_flatten #we build committor with respect to the stable state B
    phi = shooting_points_flatten[:, 0] # shape (num_epochs * K,)
    psi = shooting_points_flatten[:, 1] # shape (num_epochs * K,)
    # 1. Get the sum of the weights
    H_weights, phiedges, psiedges = np.histogram2d(phi, psi, bins=bins, weights=weights)
    # 2. Get the count of points in each bin
    H_counts, _, _ = np.histogram2d(phi, psi, bins=bins)

    # 3. Calculate the mean (Sum / Count)
    # We use a context manager to ignore the "Divide by zero" warning for empty bins
    H_mean = H_weights / (2 * H_counts)
    
    mesh = ax.pcolormesh(phiedges, psiedges, H_mean.T, cmap='jet')
    fig.colorbar(mesh, ax=ax, label='Committor probability')
    ax.set_xlabel('$\phi$')
    ax.set_ylabel('$\psi $')
    ax.set_title('Rough Committor Estimate from Shooting Points')
    plt.tight_layout()
    plt.show()
    
    if return_grid:
        return H_mean

## The definition of a simulation

In [ ]:
# ------------------------------------------------------------
# 1. Load initial UNSOLVATED structure
# ------------------------------------------------------------

pdb = PDBFile("alanine-dipeptide.pdb")

initial_topology = pdb.topology
initial_positions = pdb.positions


rotated_positions = initial_positions
# ------------------------------------------------------------
# 2. Define target dihedrals
# ------------------------------------------------------------
# Phi: C(ACE)-N(ALA)-CA(ALA)-C(ALA)
phi_atoms = [4, 6, 8, 14] 
# Psi: N(ALA)-CA(ALA)-C(ALA)-N(NME)
psi_atoms = [6, 8, 14, 16]


initial_phi = - np.pi / 3
initial_psi = 0
#This one is needed because a simulation was run for this configuration. 
#In order to reproduce the number of atoms created by the solvent, it is left here.
rotated_positions = init_rotate_dihedral(initial_topology, initial_positions, phi_atoms, np.degrees(initial_phi))
rotated_positions = init_rotate_dihedral(initial_topology, rotated_positions, psi_atoms, np.degrees(initial_psi))


# ------------------------------------------------------------
# 3. Build modeller using ROTATED structure
# ------------------------------------------------------------

modeller = Modeller(initial_topology, rotated_positions)
#modeller = Modeller(pdb.topology, pdb.positions)

# ------------------------------------------------------------
# 4. Load force field
# ------------------------------------------------------------

forcefield = ForceField(
    "amber14-all.xml",
    "amber14/tip3pfb.xml"
)

# ------------------------------------------------------------
# 5. Add solvent AFTER rotation
# ------------------------------------------------------------


modeller.addSolvent(forcefield, model="tip3p", padding=1.2 * nanometer)

# ------------------------------------------------------------
# 6. Create OpenMM system
# ------------------------------------------------------------

system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME, constraints=HBonds)

# ------------------------------------------------------------
# 7. Add barostat
# ------------------------------------------------------------

temperature = 300 * kelvin
pressure = 1 * bar

system.addForce(
    MonteCarloBarostat(
        pressure,
        temperature
    )
)

# ------------------------------------------------------------
# 8. Create simulation
# ------------------------------------------------------------
integrator = LangevinIntegrator(temperature, 1 / picosecond, 1 * femtoseconds)
simulation = Simulation(
    modeller.topology,
    system,
    integrator
)
simulation.context.setPositions(modeller.positions)
simulation.minimizeEnergy()

simulation.reporters = []

interval = 30 #There will be no issue as long as the check interval in TPS and the writing interval are the same.
scaling_factor = 1/interval #PROBLEM
h5_file_path = 'shooting_data_interval1.h5'
active_h5_file = 'model_training.h5'

my_reporter = md.reporters.HDF5Reporter(
    active_h5_file, 
    interval,
    coordinates=True,     # Replaces DCDReporter
    velocities=True,      # Saves particle velocities
    potentialEnergy=True, # Replaces StateDataReporter for energy
    temperature=True      # Replaces StateDataReporter for temp
)
simulation.reporters.append(my_reporter)
topology = md.load_topology(h5_file_path)






With the continuing TPS

Data for LAM = 1

In [ ]:

# ── device selection ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}\n")

# ── hyper-parameters ────────────────────────────────────────────────────
INPUT_DIM   = 2      # phi, psi. 
HIDDEN_DIM  =  [192, 224, 128, 64, 32] # number of neurons in each hidden layer [160, 176, 64, 32], [160, 176, 144, 48, 32], [192, 224, 160, 96, 64, 16]
K           = 70    # shooting algorithm trials
LAM         =  1 #5e-4
LR          = 2e-3
N_ITER      = 75
ALPHA_THRESH = (1 - (K - np.log10(K) + 2) / K)**2 # only update if α_eff > 1 - (K-2)/K, i.e. if the fail is max 2 transition paths per iteration 

source_h5_file_path = 'shooting_data_interval1.h5'

# ── model + optimiser ───────────────────────────────────────────────────
source_transition_indices = np.load('transition_indices_DATA.npy') 

model = CommittorNet(INPUT_DIM, HIDDEN_DIM).to(device) 
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',      # we minimize loss
    factor=0.5,      # LR ← LR / 2
    patience=5,     # wait 5 epochs with no improvement
)

sampled_indices = torch.randint(0, source_transition_indices.shape[0], (1,)) #randomly sample K transition paths from the reference file
path_idx = source_transition_indices[sampled_indices.item()]
begin_index = [path_idx[0], path_idx[2]]
end_index = [path_idx[1], path_idx[3]]
initial_path_length = path_idx[1] - path_idx[0] +  path_idx[3] - path_idx[2]
copy_slice_h5_to_reporter(simulation.reporters[0], source_h5_file_path, begin_index, end_index)
#extract initial phi psi values
with h5py.File(active_h5_file, 'r') as f:
    path_coord = f['coordinates'][0 : initial_path_length]

phi_rad = numpy_dihedral_batch(path_coord, phi_atoms) #(initial_path_length,)
psi_rad = numpy_dihedral_batch(path_coord, psi_atoms) #(initial_path_length,)
path = np.column_stack((phi_rad, psi_rad)) #(initial_path_length, 2)
phis_psis_tensor = torch.tensor(path, dtype=torch.float32).to(device)  # (path_len, input_dim)
# ── Select first ──────────────
with torch.no_grad():
    N_s       = model(phis_psis_tensor)                                # (path_len,)
    sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (path_len,) on CPU
los = np.empty(N_ITER)
n_a_data = np.empty((N_ITER, K))
angles_data = np.empty((N_ITER, K, 2))


for it_idx in range(N_ITER):
    angles_list  = []
    n_A_arr      = np.empty(K, dtype=np.int32)
    n_B_arr      = np.empty(K, dtype=np.int32)
    n_tps_gen    = 0

    writing_start_index = initial_path_length
    initial_transition_indices = [0, initial_path_length]
    # ── Run K two-way shooting attempts, collect results ─────────────────
    for trial in range(K):
        indices, n_a, n_b, angles_config = Create_a_transition_path_with_shooting(
            simulation, 
            initial_transition_indices, 
            active_h5_file, active_h5_file, 
            interval = interval,
            max_steps = 10**5,
            return_nA_nB_pHipSi=True,
            probability_distribution = sel_probs,
            scaling_factor=scaling_factor)
        if  (indices[0] is not None):
            #set new initial_transition_indices
            #This is a modified version for the case of phi > 0 because we have rest frames 
            initial_transition_indices = [
                writing_start_index + indices[3] - indices[0] - indices[1],  #Index of the first half end 
                writing_start_index + indices[3] - indices[1], #As long as there is no gap, this is correct. 
                writing_start_index + indices[3] - indices[1], #Index of the second half beginning
                writing_start_index + indices[3] - indices[2]]
            
            scaling_factor = 1
            n_tps_gen += 1
            #recalculate the probability distribution
            with h5py.File(active_h5_file, 'r') as f:
                first_half = f['coordinates'][initial_transition_indices[0] : initial_transition_indices[1]]
                second_half = f['coordinates'][initial_transition_indices[2] : initial_transition_indices[3]]
            #I do not revert here because there are 4 indices
            coordinates = np.concatenate((first_half, second_half), axis=0)
            phi_rad = numpy_dihedral_batch(coordinates, phi_atoms) #(initial_path_length,)
            psi_rad = numpy_dihedral_batch(coordinates, psi_atoms) #(initial_path_length,)

            #I WANT TO BE REALY SURE
            if np.sum(angles_in_stable_region(phi_rad, psi_rad)) != 2:
                print("ALARM")
                print("indices:", initial_transition_indices)
                print("len", path.shape[0])
                print("Path before")
                plot_ramachandran(_, "ram2", highlight_stable=True, phi_psi_traj=path )
                raise ValueError

            path = np.column_stack((phi_rad, psi_rad)) #(initial_path_length, 2)
            if path.shape[0] == 0:
                print(initial_transition_indices)
                raise RuntimeError("The length of the transition path becomes zero.")
            phis_psis_tensor = torch.tensor(path, dtype=torch.float32).to(device)       # (path_len, input_dim)
            # ── Select first ──────────────
            with torch.no_grad():
                N_s       = model(phis_psis_tensor)                                      # (path_len,)
                sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (path_len,) on CPU
        
        
        writing_start_index += indices[3]
        n_A_arr[trial] = n_a
        n_B_arr[trial] = n_b
        angles_list.append(angles_config)
            

            
            
    """if (it_idx+1)%(N_ITER//5) == 0:
        #Little test
        print("Number of iteration:", it_idx)
        plot_ramachandran(_, "ram2", highlight_stable=True, phi_psi_traj=path )
        print("The number of transition points in a stable state:", np.sum(angles_in_stable_region(phi_rad, psi_rad)))
        #Little test"""

    # ── Build batched tensors (K, ...) and run one training step ─────────
    
    angles_configs = torch.tensor(np.stack(angles_list), dtype=torch.float32)   # (K, input_dim)
    n_A_tensor     = torch.tensor(n_A_arr, dtype=torch.int)                     # (K,)
    n_B_tensor     = torch.tensor(n_B_arr, dtype=torch.int)                     # (K,)

    n_a_data[it_idx] = n_A_arr
    angles_data[it_idx] = angles_list


    res = train_step(
        model, angles_configs, n_A_tensor, n_B_tensor,
        n_tps_gen=n_tps_gen,
        optimizer=optimizer,
        alpha_eff_threshold=ALPHA_THRESH,
        device=device,
        scheduler=scheduler,
    )
    los[it_idx] = res['alpha_eff']

    sliced_data = {}
    
    with h5py.File(active_h5_file, 'r') as f:
        for key in f.keys():
            if len(initial_transition_indices)==2: #In case of no successful TPS
                sliced_data[key] = f[key][initial_transition_indices[0] : initial_transition_indices[1]]
            else:
                first_half = f[key][initial_transition_indices[0] : initial_transition_indices[1]]
                first_half = first_half[::-1] #I revert here because I need the whole path within 0:len
                second_half = f[key][initial_transition_indices[2] : initial_transition_indices[3]]
                sliced_data[key] = np.concatenate((first_half, second_half), axis=0)
    
    # Find and remove the HDF5Reporter from simulation 
    my_reporter.close()
    simulation.reporters.remove(my_reporter)
    my_reporter = md.reporters.HDF5Reporter(
        active_h5_file, 
        interval,
        coordinates=True,     # Replaces DCDReporter
        velocities=True,      # Saves particle velocities
        potentialEnergy=True, # Replaces StateDataReporter for energy
        temperature=True      # Replaces StateDataReporter for temp
    )
    
    traj_writer = my_reporter._traj_file
    traj_writer.write(
        cell_angles = sliced_data['cell_angles'], 
        cell_lengths = sliced_data['cell_lengths'], 
        coordinates = sliced_data['coordinates'], 
        kineticEnergy = sliced_data['kineticEnergy'], 
        potentialEnergy = sliced_data['potentialEnergy'], 
        temperature = sliced_data['temperature'], 
        time = sliced_data['time'], 
        velocities = sliced_data['velocities']
    )
    if len(initial_transition_indices) == 2:
        initial_path_length = initial_transition_indices[1] - initial_transition_indices[0]
    else:
        initial_path_length = initial_transition_indices[1] - initial_transition_indices[0] +  initial_transition_indices[3] - initial_transition_indices[2]
    
    traj_writer.flush()
    simulation.reporters.append(my_reporter)


#plotting the learning curve
plt.figure(figsize=(8, 6))
plt.plot(range(N_ITER), los, 'x-')
plt.axhline(ALPHA_THRESH, color='red', linestyle='--')
plt.title(f'Learning curve for {N_ITER} iterations')
plt.xlabel('Iteration')
plt.ylabel('Effective Alpha')
plt.tight_layout()
plt.show()

model.save_weights('committor_model_alanine_dipeptide_LAM_1_[192, 224, 128, 64, 32]')
del simulation

import tables
tables.file._open_files.close_all()
evaluate_model(model)
plot_shooting_point_density(angles_data)
plot_raugh_commitor(angles_data, n_a_data)


With a new TPS for each iteration. 

In [ ]:

# ── device selection ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}\n")

# ── hyper-parameters ────────────────────────────────────────────────────
INPUT_DIM   = 2      # phi, psi. 
HIDDEN_DIM  =  [192, 224, 128, 64, 32] # number of neurons in each hidden layer [160, 176, 64, 32], [160, 176, 144, 48, 32]
K           = 40    # shooting algorithm trials
LAM         = 5e-4
LR          = 2e-3
N_ITER      = 75
ALPHA_THRESH = (1 - (K - np.log10(K) + 2) / K)**2 # only update if α_eff > 1 - (K-2)/K, i.e. if the fail is max 2 transition paths per iteration 

source_h5_file_path = 'shooting_data_interval1.h5'

# ── model + optimiser ───────────────────────────────────────────────────
source_transition_indices = np.load('transition_indices_DATA.npy') 

model = CommittorNet(INPUT_DIM, HIDDEN_DIM).to(device) 
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',      # we minimize loss
    factor=0.5,      # LR ← LR / 2
    patience=30,     # wait 30 epochs with no improvement
)

sampled_indices = torch.randint(0, source_transition_indices.shape[0], (N_ITER,)) #randomly sample K transition paths from the reference file
los = np.empty(N_ITER)



for it_idx, it in enumerate(sampled_indices):
    path_idx = source_transition_indices[it.item()]
    begin_index = [path_idx[0], path_idx[2]]
    end_index = [path_idx[1], path_idx[3]]
    initial_path_length = path_idx[1] - path_idx[0] +  path_idx[3] - path_idx[2]
    copy_slice_h5_to_reporter(simulation.reporters[0], source_h5_file_path, begin_index, end_index)
    #extract initial phi psi values
    with h5py.File(active_h5_file, 'r') as f:
        path_coord = f['coordinates'][0 : initial_path_length]

    phi_rad = numpy_dihedral_batch(path_coord, phi_atoms) #(initial_path_length,)
    psi_rad = numpy_dihedral_batch(path_coord, psi_atoms) #(initial_path_length,)
    path = np.column_stack((phi_rad, psi_rad)) #(initial_path_length, 2)
    phis_psis_tensor = torch.tensor(path, dtype=torch.float32).to(device)  # (path_len, input_dim)
    # ── Select first ──────────────
    with torch.no_grad():
        N_s       = model(phis_psis_tensor)                                # (path_len,)
        sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (path_len,) on CPU

    angles_list  = []
    n_A_arr      = np.empty(K, dtype=np.int32)
    n_B_arr      = np.empty(K, dtype=np.int32)
    n_tps_gen    = 0
    writing_start_index = initial_path_length
    initial_transition_indices = [0, initial_path_length]
    # ── Run K two-way shooting attempts, collect results ─────────────────
    for trial in range(K):
        indices, n_a, n_b, angles_config = Create_a_transition_path_with_shooting(
            simulation, 
            initial_transition_indices, 
            active_h5_file, active_h5_file, 
            interval = interval,
            return_nA_nB_pHipSi=True,
            probability_distribution = sel_probs,
            scaling_factor=scaling_factor)
        if not (indices[0] is None):
            #set new initial_transition_indices
            #This is a modified version for the case of phi > 0 because we have rest frames 
            initial_transition_indices = [
                writing_start_index + indices[3] - indices[0] - indices[1],  #Index of the first half end 
                writing_start_index + indices[3] - indices[1], #As long as there is no gap, this is correct. 
                writing_start_index + indices[3] - indices[1], #Index of the second half beginning
                writing_start_index + indices[3] - indices[2]]
            n_tps_gen += 1
            #recalculate the probability distribution
            with h5py.File(active_h5_file, 'r') as f:
                first_half = f['coordinates'][initial_transition_indices[0] : initial_transition_indices[1]]
                first_half = first_half[::-1]
                second_half = f['coordinates'][initial_transition_indices[2] : initial_transition_indices[3]]

            coordinates = np.concatenate((first_half, second_half), axis=0)
            phi_rad = numpy_dihedral_batch(coordinates, phi_atoms) #(initial_path_length,)
            psi_rad = numpy_dihedral_batch(coordinates, psi_atoms) #(initial_path_length,)
            path = np.column_stack((phi_rad, psi_rad)) #(initial_path_length, 2)
            phis_psis_tensor = torch.tensor(path, dtype=torch.float32).to(device)       # (path_len, input_dim)
            # ── Select first ──────────────
            with torch.no_grad():
                N_s       = model(phis_psis_tensor)                                      # (path_len,)
                sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (path_len,) on CPU

        
        writing_start_index += indices[3]
        n_A_arr[trial] = n_a
        n_B_arr[trial] = n_b
        angles_list.append(angles_config)
            
    """Little test
    if it_idx%10 == 0:
        plot_ramachandran(_, "ram2", highlight_stable=True, phi_psi_traj=path )
        print("The number of transition points in a stable state:", np.sum(angles_in_stable_region(phi_rad, psi_rad)))
        """
        # ── Build batched tensors (K, ...) and run one training step ─────────
    
    angles_configs = torch.tensor(np.stack(angles_list), dtype=torch.float32)   # (K, input_dim)
    n_A_tensor     = torch.tensor(n_A_arr, dtype=torch.int)                     # (K,)
    n_B_tensor     = torch.tensor(n_B_arr, dtype=torch.int)                     # (K,)



    res = train_step(
        model, angles_configs, n_A_tensor, n_B_tensor,
        n_tps_gen=n_tps_gen,
        optimizer=optimizer,
        alpha_eff_threshold=ALPHA_THRESH,
        device=device,
        scheduler=scheduler,
    )
    los[it_idx] = res['alpha_eff']

    # Find and remove the HDF5Reporter from simulation 
    my_reporter.close()
    simulation.reporters.remove(my_reporter)
    my_reporter = md.reporters.HDF5Reporter(
        active_h5_file, 
        interval,
        coordinates=True,     # Replaces DCDReporter
        velocities=True,      # Saves particle velocities
        potentialEnergy=True, # Replaces StateDataReporter for energy
        temperature=True      # Replaces StateDataReporter for temp
    )
    simulation.reporters.append(my_reporter)

#plotting the learning curve
plt.figure(figsize=(8, 6))
plt.plot(range(N_ITER), los, 'x-')
plt.axhline(ALPHA_THRESH, color='red', linestyle='--')
plt.title(f'Learning curve for {N_ITER} iterations')
plt.xlabel('Iteration')
plt.ylabel('Effective Alpha')
plt.tight_layout()
plt.show()

import tables
tables.file._open_files.close_all()

del simulation
evaluate_model(model)

In [ ]:
model.save_weights('committor_model_alanine_dipeptide_LAM_5e-4_[192, 224, 128, 64, 32]')

# Optuna optimisation

In [ ]:
import optuna
from optuna.trial import Trial
from torch.utils.data import DataLoader

def objective(
    simulation,
    trial: Trial,
    train_loader: DataLoader, #Removed val_loader because calculating this was too expensive.
    input_dim: int,
    device: torch.device,
    #ALPHA_THRESH: float,
    #batch_size: int,

) -> float:
    """
    Optuna objective function.

    Returns:
        Validation MAE on normalized targets (lower is better).
    """
    #a small crutch 
    my_reporter = simulation.reporters[0]
    my_reporter.close()
    simulation.reporters.remove(my_reporter)
    my_reporter = md.reporters.HDF5Reporter(
    active_h5_file, 
    interval,
    coordinates=True,     # Replaces DCDReporter
    velocities=True,      # Saves particle velocities
    potentialEnergy=True, # Replaces StateDataReporter for energy
    temperature=True      # Replaces StateDataReporter for temp
    )
    simulation.reporters.append(my_reporter)
    #a small crutch PAY ATTENTION TO THIS

    # Hyperparameters to tune
    learning_rate = 1e-3
    #hidden_layers = trial.suggest_int("hidden_layers", 5, 7)
    batch_size = 30
    ALPHA_THRESH = trial.suggest_float("ALPHA_THRESH", 0.001, 0.2) #for 100 --> 4, 10 -->3,  mistakes per iteration are allowed
    #architecture = {}
    #for i in range(hidden_layers):
    #    architecture[f"hidden_width_{i}"] = trial.suggest_int(f"hidden_width_{i}", 16, 192, step=16)
    num_epochs = 50
    LAM = 5e-4
    HIDDEN_DIM  =trial.suggest_categorical("Architecture", [[128, 192, 64], [192, 224, 160, 64, 32]])
    #HIDDEN_DIM = list(architecture.values())
    model = CommittorNet(input_dim, HIDDEN_DIM).to(device) 
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    source_transition_indices = np.load('transition_indices_DATA.npy') 
    sampled_indices = torch.randint(0, source_transition_indices.shape[0], (1,)) #randomly sample a transition path from the reference file
    path_idx = source_transition_indices[sampled_indices.item()]
    begin_index = [path_idx[0], path_idx[2]]
    end_index = [path_idx[1], path_idx[3]]
    initial_path_length = path_idx[1] - path_idx[0] +  path_idx[3] - path_idx[2]
    copy_slice_h5_to_reporter(simulation.reporters[0], train_loader, begin_index, end_index)
    #extract initial phi psi values
    with h5py.File(active_h5_file, 'r') as f:
        path_coord = f['coordinates'][0 : initial_path_length]

    phi_rad = numpy_dihedral_batch(path_coord, phi_atoms) #(initial_path_length,)
    psi_rad = numpy_dihedral_batch(path_coord, psi_atoms) #(initial_path_length,)
    path = np.column_stack((phi_rad, psi_rad)) #(initial_path_length, 2)
    phis_psis_tensor = torch.tensor(path, dtype=torch.float32).to(device)  # (path_len, input_dim)
    # ── Select first ──────────────
    with torch.no_grad():
        N_s       = model(phis_psis_tensor)                                # (path_len,)
        sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (path_len,) on CPU
    los = np.empty(num_epochs)

    for epoch in range(num_epochs):
        angles_list  = []
        n_A_arr      = np.empty(batch_size, dtype=np.int32)
        n_B_arr      = np.empty(batch_size, dtype=np.int32)
        n_tps_gen    = 0

        writing_start_index = initial_path_length
        initial_transition_indices = [0, initial_path_length]
        # ── Run shooting attempts, collect results ─────────────────
        for shoot_trial in range(batch_size):
            indices, n_a, n_b, angles_config = Create_a_transition_path_with_shooting(
                simulation, 
                initial_transition_indices, 
                active_h5_file, active_h5_file, 
                interval = interval,
                max_steps = 5*10**5,
                return_nA_nB_pHipSi=True,
                probability_distribution = sel_probs,)
            
            if not (indices[0] is None):
                #set new initial_transition_indices
                initial_transition_indices = [
                writing_start_index + indices[3] - indices[0] - indices[1],  #Index of the first half end 
                writing_start_index + indices[3] - indices[1], #As long as there is no gap, this is correct. 
                writing_start_index + indices[3] - indices[1], #Index of the second half beginning
                writing_start_index + indices[3] - indices[2]]
                n_tps_gen += 1
                #recalculate the probability distribution
                with h5py.File(active_h5_file, 'r') as f:
                    first_half = f['coordinates'][initial_transition_indices[0] : initial_transition_indices[1]]
                    second_half = f['coordinates'][initial_transition_indices[2] : initial_transition_indices[3]]
                #I do not revert here because there are 4 indices
                coordinates = np.concatenate((first_half, second_half), axis=0)
                phi_rad = numpy_dihedral_batch(coordinates, phi_atoms) #(initial_path_length,)
                psi_rad = numpy_dihedral_batch(coordinates, psi_atoms) #(initial_path_length,)

                #I WANT TO BE REALY SURE
                if np.sum(angles_in_stable_region(phi_rad, psi_rad)) != 2:
                    print("ALARM")
                    print("indices:", initial_transition_indices)
                    print("len", path.shape[0])
                    print("Path before")
                    plot_ramachandran(_, "ram2", highlight_stable=True, phi_psi_traj=path )
                    raise ValueError

                path = np.column_stack((phi_rad, psi_rad)) #(initial_path_length, 2)
                phis_psis_tensor = torch.tensor(path, dtype=torch.float32).to(device)       # (path_len, input_dim)
                # ── Select first ──────────────
                with torch.no_grad():
                    N_s       = model(phis_psis_tensor)                                      # (path_len,)
                    sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (path_len,) on CPU
            writing_start_index += indices[3]
            n_A_arr[shoot_trial] = n_a
            n_B_arr[shoot_trial] = n_b
            angles_list.append(angles_config)

        # ── Build batched tensors (batch_size, ...) and run one training step ─────────
        angles_configs = torch.tensor(np.stack(angles_list), dtype=torch.float32)   # (batch_size, input_dim)
        n_A_tensor     = torch.tensor(n_A_arr, dtype=torch.int)                     # (batch_size,)
        n_B_tensor     = torch.tensor(n_B_arr, dtype=torch.int)                     # (batch_size,)

        res = train_step(
        model, angles_configs, n_A_tensor, n_B_tensor,
        n_tps_gen=n_tps_gen,
        optimizer=optimizer,
        alpha_eff_threshold=ALPHA_THRESH,
        device=device,
        )
        los[epoch] = res['alpha_eff']

        # Report intermediate metric for pruning
        trial.report(los[epoch], step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        #Preparation for the next epoch
        sliced_data = {}
        
        with h5py.File(active_h5_file, 'r') as f:
            for key in f.keys():
                if len(initial_transition_indices)==2: #In case of no successful TPS
                    sliced_data[key] = f[key][initial_transition_indices[0] : initial_transition_indices[1]]
                else:
                    first_half = f[key][initial_transition_indices[0] : initial_transition_indices[1]]
                    first_half = first_half[::-1] #I revert here because I need the whole path within 0:len
                    second_half = f[key][initial_transition_indices[2] : initial_transition_indices[3]]
                    sliced_data[key] = np.concatenate((first_half, second_half), axis=0)
        
        # Find and remove the HDF5Reporter from simulation 
        my_reporter.close()
        simulation.reporters.remove(my_reporter)
        my_reporter = md.reporters.HDF5Reporter(
            active_h5_file, 
            interval,
            coordinates=True,     # Replaces DCDReporter
            velocities=True,      # Saves particle velocities
            potentialEnergy=True, # Replaces StateDataReporter for energy
            temperature=True      # Replaces StateDataReporter for temp
        )
        
        traj_writer = my_reporter._traj_file
        traj_writer.write(
            cell_angles = sliced_data['cell_angles'], 
            cell_lengths = sliced_data['cell_lengths'], 
            coordinates = sliced_data['coordinates'], 
            kineticEnergy = sliced_data['kineticEnergy'], 
            potentialEnergy = sliced_data['potentialEnergy'], 
            temperature = sliced_data['temperature'], 
            time = sliced_data['time'], 
            velocities = sliced_data['velocities']
        )
        #Just in case there was no successful TPS
        if len(initial_transition_indices) == 2:
            initial_path_length = initial_transition_indices[1] - initial_transition_indices[0]
        else:
            initial_path_length = initial_transition_indices[1] - initial_transition_indices[0] +  initial_transition_indices[3] - initial_transition_indices[2]
        
        traj_writer.flush()
        simulation.reporters.append(my_reporter)
    
    return np.mean(los)  # Return the average loss over epochs for this trial


# ── Optuna study setup ────────────────────────────────────────────────────
pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5,
        interval_steps=1,
    )

study = optuna.create_study(
        direction="minimize",  # we want to minimize the validation MAE
        pruner=pruner,
        study_name="pytorch_mlp_optimization",
    )

INPUT_DIM   = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}\n")
source_h5_file_path = 'shooting_data_interval1.h5'
study.optimize(
        lambda trial: objective(
            simulation = simulation,
            trial=trial,
            train_loader = source_h5_file_path,
            input_dim=INPUT_DIM,
            device=device,
        ),
        n_trials=150,
        timeout=7200,
        show_progress_bar=True,
    )

print("\nOptimization complete.")
print(f"Best trial number: {study.best_trial.number}")
print(f"Best validation MAE: {study.best_value:.6f}")
print("Best hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")
